# H&M Personalized Fashion Recommendations 데이터셋을 지식그래프용 CSV로 변환하기

H&M 데이터셋: https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations/data

**데이터셋 구성:**
- `articles.csv`: 제품 정보 (약 105,542개)
- `customers.csv`: 고객 정보 (약 1,371,980개)
- `transactions_train.csv`: 구매 이력 (약 31,788,324개)

In [1]:
import pandas as pd
import os

articles = pd.read_csv('./datasets/articles.csv')
customers = pd.read_csv('./datasets/customers.csv')
transactions = pd.read_csv('./datasets/transactions_train.csv')

print(f"Articles: {len(articles):,}개")
print(f"Customers: {len(customers):,}개")
print(f"Transactions: {len(transactions):,}개")

Articles: 105,542개
Customers: 1,371,980개
Transactions: 31,788,324개


### 데이터 샘플링 옵션(선택)

In [15]:
SAMPLE_SIZE = 500000

# 최근 데이터만 샘플링 (시간순)
transactions = transactions.sort_values('t_dat', ascending=False).head(SAMPLE_SIZE)

# 샘플링된 거래에 포함된 고객/제품만 필터링
sampled_customer_ids = transactions['customer_id'].unique()
sampled_article_ids = transactions['article_id'].unique()

customers = customers[customers['customer_id'].isin(sampled_customer_ids)]
articles = articles[articles['article_id'].isin(sampled_article_ids)]

print(f"샘플 데이터 준비 완료:")
print(f"  • Transactions: {len(transactions):,}개")
print(f"  • Customers: {len(customers):,}개")
print(f"  • Articles: {len(articles):,}개")

샘플 데이터 준비 완료:
  • Transactions: 500,000개
  • Customers: 129,284개
  • Articles: 22,694개


## 1. 데이터 탐색

In [16]:
pd.set_option('display.max_column', None)

In [17]:
# Articles 데이터 미리보기
articles.head()

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,3,Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
6,111565001,111565,20 den 1p Stockings,304,Underwear Tights,Socks & Tights,1010016,Solid,9,Black,4,Dark,5,Black,3608,Tights basic,B,Lingeries/Tights,1,Ladieswear,62,"Womens Nightwear, Socks & Tigh",1021,Socks and Tights,"Semi shiny nylon stockings with a wide, reinfo..."
8,111586001,111586,Shape Up 30 den 1p Tights,273,Leggings/Tights,Garment Lower body,1010016,Solid,9,Black,4,Dark,5,Black,3608,Tights basic,B,Lingeries/Tights,1,Ladieswear,62,"Womens Nightwear, Socks & Tigh",1021,Socks and Tights,Tights with built-in support to lift the botto...
9,111593001,111593,Support 40 den 1p Tights,304,Underwear Tights,Socks & Tights,1010016,Solid,9,Black,4,Dark,5,Black,3608,Tights basic,B,Lingeries/Tights,1,Ladieswear,62,"Womens Nightwear, Socks & Tigh",1021,Socks and Tights,"Semi shiny tights that shape the tummy, thighs..."
10,111609001,111609,200 den 1p Tights,304,Underwear Tights,Socks & Tights,1010016,Solid,9,Black,4,Dark,5,Black,3608,Tights basic,B,Lingeries/Tights,1,Ladieswear,62,"Womens Nightwear, Socks & Tigh",1021,Socks and Tights,Opaque matt tights. 200 denier.


In [18]:
# Customers 데이터 미리보기
customers.head()

,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,NaN,NaN,ACTIVE,NONE,24.0,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...
6,0000757967448a6cb83efb3ea7a3fb9d418ac7adf2379d...,NaN,NaN,ACTIVE,NONE,20.0,fe7b8e2b3fafb89ca90db17ffeeae0fd29b795d803f749...
38,0001d44dbe7f6c4b35200abdb052c77a87596fe1bdcc37...,1.0,1.0,ACTIVE,Regularly,44.0,930b19ae7db8abb5a27f4da10217755a7305b4c452f5e0...
80,00039306476aaf41a07fed942884f16b30abfa83a2a8be...,NaN,NaN,ACTIVE,NONE,27.0,43cbf97df3d118b937551fb21a08d513bfb2e58223315f...
86,0003e867a930d0d6842f923d6ba7c9b77aba33fe2a0fbf...,1.0,1.0,ACTIVE,Regularly,33.0,d647e4ede3d0eb4ce0750440a110350b5f4c758165d89d...


In [19]:
# Transactions 데이터 미리보기
transactions.head(10)

,t_dat,customer_id,article_id,price,sales_channel_id
31788323,2020-09-22,fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20...,898573003,0.033881,2
31766498,2020-09-22,54e8ebd39543b5a4d69c3e7d79977558d2a606e6540ba0...,573085043,0.033881,2
31766404,2020-09-22,544094a3ab237bf18d7bda9c2265218de4320ce795775e...,839332001,0.067780,1
31766405,2020-09-22,544094a3ab237bf18d7bda9c2265218de4320ce795775e...,865938002,0.025407,1
31766406,2020-09-22,544094a3ab237bf18d7bda9c2265218de4320ce795775e...,770315017,0.016932,1
31766407,2020-09-22,544094a3ab237bf18d7bda9c2265218de4320ce795775e...,839332008,0.067780,1
31766408,2020-09-22,54417f21fc7142adf6ab87d63ed3fa925bae4825a0c869...,898692006,0.067780,2
31766409,2020-09-22,54417f21fc7142adf6ab87d63ed3fa925bae4825a0c869...,873274002,0.050831,2
31766410,2020-09-22,5445a8017a614f74b7caf15c4b17d5b5a696a745de082b...,884535001,0.050831,2
31766411,2020-09-22,5445a8017a614f74b7caf15c4b17d5b5a696a745de082b...,923010002,0.016932,2


In [20]:
print(f"기간: {transactions['t_dat'].min()} ~ {transactions['t_dat'].max()}")
print(f"고유 고객 수: {transactions['customer_id'].nunique():,}")
print(f"고유 제품 수: {transactions['article_id'].nunique():,}")

기간: 2020-09-08 ~ 2020-09-22
고유 고객 수: 129,284
고유 제품 수: 22,694


In [21]:
# article_id 별 구매한 고유 사용자 수 확인
article_purchase_stats = transactions.groupby('article_id').agg({
    'customer_id': 'nunique',  # 고유 고객 수
    't_dat': 'count'  # 총 구매 횟수
}).rename(columns={
    'customer_id': 'unique_customers',
    't_dat': 'total_purchases'
}).sort_values('unique_customers', ascending=False)

# 제품 정보와 조인
article_purchase_stats = article_purchase_stats.merge(
    articles[['article_id', 'prod_name', 'product_type_name', 'colour_group_name']],
    left_index=True,
    right_on='article_id',
    how='left'
)

print(f"=== 가장 많은 고객이 구매한 제품 TOP 20 ===\n")
print(article_purchase_stats.head(20).to_string(index=False))

# 구매자 수 분포
print(f"\n=== 구매자 수 분포 ===")
print(f"  • 1명만 구매: {(article_purchase_stats['unique_customers'] == 1).sum():,}개 제품")
print(f"  • 2-10명 구매: {((article_purchase_stats['unique_customers'] >= 2) & (article_purchase_stats['unique_customers'] <= 10)).sum():,}개 제품")
print(f"  • 11-100명 구매: {((article_purchase_stats['unique_customers'] >= 11) & (article_purchase_stats['unique_customers'] <= 100)).sum():,}개 제품")
print(f"  • 100명 이상 구매: {(article_purchase_stats['unique_customers'] > 100).sum():,}개 제품")

=== 가장 많은 고객이 구매한 제품 TOP 20 ===

 unique_customers  total_purchases  article_id                 prod_name product_type_name colour_group_name
             1476             1822   909370001         FF PI Haley dress             Dress         Off White
             1367             1560   924243001                   Ohlsson           Sweater             Beige
             1217             1343   918522001         Jackie cable vest           Sweater         Off White
             1028             1216   865799006            Pink HW barrel          Trousers       Light Beige
              916             1147   751471001       Pluto RW slacks (1)          Trousers             Black
              904              997   918292001 STRONG HW seamless tights   Leggings/Tights             Black
              891             1049   923758001                   Vanessa           Unknown             White
              875              949   915529003                   Liliana           Sweater     

## 2. 그래프 스키마 설계

**노드:**
- **Article**: 제품 (article_id, prod_name, price, color, etc.)
- **Customer**: 고객 (customer_id, age, postal_code, etc.)
- **ProductType**: 제품 유형 (product_type_no, product_type_name)
- **ProductGroup**: 제품 그룹 (product_group_name)
- **ColourGroup**: 색상 그룹 (colour_group_code, colour_group_name)
- **Department**: 부서 (department_no, department_name)
- **Section**: 섹션 (section_no, section_name)
- **GarmentGroup**: 의류 그룹 (garment_group_no, garment_group_name)

**관계:**
- Customer -[:PURCHASED {date, price, sales_channel_id}]-> Article
- Article -[:OF_TYPE]-> ProductType
- Article -[:IN_GROUP]-> ProductGroup
- Article -[:HAS_COLOUR]-> ColourGroup
- Article -[:IN_DEPARTMENT]-> Department
- Article -[:IN_SECTION]-> Section
- Article -[:IN_GARMENT_GROUP]-> GarmentGroup

## 3. 노드 CSV 생성

In [22]:
# 1) Article 노드
print("1/8 Article 노드 생성 중...")
nodes_article = articles[[
    'article_id',
    'product_code',
    'prod_name',
    'product_type_no',
    'product_type_name',
    'product_group_name',
    'graphical_appearance_no',
    'graphical_appearance_name',
    'colour_group_code',
    'colour_group_name',
    'perceived_colour_value_id',
    'perceived_colour_value_name',
    'perceived_colour_master_id',
    'perceived_colour_master_name',
    'department_no',
    'department_name',
    'index_code',
    'index_name',
    'index_group_no',
    'index_group_name',
    'section_no',
    'section_name',
    'garment_group_no',
    'garment_group_name',
    'detail_desc'
]].copy()

print(f"  ✓ Article 노드: {len(nodes_article):,}개")
nodes_article.head(3)

1/8 Article 노드 생성 중...
  ✓ Article 노드: 22,694개


,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,3,Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
6,111565001,111565,20 den 1p Stockings,304,Underwear Tights,Socks & Tights,1010016,Solid,9,Black,4,Dark,5,Black,3608,Tights basic,B,Lingeries/Tights,1,Ladieswear,62,"Womens Nightwear, Socks & Tigh",1021,Socks and Tights,"Semi shiny nylon stockings with a wide, reinfo..."
8,111586001,111586,Shape Up 30 den 1p Tights,273,Leggings/Tights,Garment Lower body,1010016,Solid,9,Black,4,Dark,5,Black,3608,Tights basic,B,Lingeries/Tights,1,Ladieswear,62,"Womens Nightwear, Socks & Tigh",1021,Socks and Tights,Tights with built-in support to lift the botto...


In [23]:
# 2) Customer 노드
print("2/8 Customer 노드 생성 중...")
nodes_customer = customers[[
    'customer_id',
    'FN',
    'Active',
    'club_member_status',
    'fashion_news_frequency',
    'age',
    'postal_code'
]].copy()

print(f"  ✓ Customer 노드: {len(nodes_customer):,}개")
nodes_customer.head(3)

2/8 Customer 노드 생성 중...
  ✓ Customer 노드: 129,284개


,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,NaN,NaN,ACTIVE,NONE,24.0,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...
6,0000757967448a6cb83efb3ea7a3fb9d418ac7adf2379d...,NaN,NaN,ACTIVE,NONE,20.0,fe7b8e2b3fafb89ca90db17ffeeae0fd29b795d803f749...
38,0001d44dbe7f6c4b35200abdb052c77a87596fe1bdcc37...,1.0,1.0,ACTIVE,Regularly,44.0,930b19ae7db8abb5a27f4da10217755a7305b4c452f5e0...


In [24]:
# 3) 분류 노드들 생성
print("3/8 ProductType 노드 생성 중...")
nodes_product_type = articles[['product_type_no', 'product_type_name']].drop_duplicates()
print(f"  ✓ ProductType 노드: {len(nodes_product_type):,}개")

print("4/8 ProductGroup 노드 생성 중...")
nodes_product_group = articles[['product_group_name']].drop_duplicates()
nodes_product_group = nodes_product_group.rename(columns={'product_group_name': 'name'})
print(f"  ✓ ProductGroup 노드: {len(nodes_product_group):,}개")

print("5/8 ColourGroup 노드 생성 중...")
nodes_colour_group = articles[['colour_group_code', 'colour_group_name']].drop_duplicates()
print(f"  ✓ ColourGroup 노드: {len(nodes_colour_group):,}개")

print("6/8 Department 노드 생성 중...")
nodes_department = articles[['department_no', 'department_name']].drop_duplicates()
print(f"  ✓ Department 노드: {len(nodes_department):,}개")

print("7/8 Section 노드 생성 중...")
nodes_section = articles[['section_no', 'section_name']].drop_duplicates()
print(f"  ✓ Section 노드: {len(nodes_section):,}개")

print("8/8 GarmentGroup 노드 생성 중...")
nodes_garment_group = articles[['garment_group_no', 'garment_group_name']].drop_duplicates()
print(f"  ✓ GarmentGroup 노드: {len(nodes_garment_group):,}개")

print("\n노드 생성 완료!")

3/8 ProductType 노드 생성 중...
  ✓ ProductType 노드: 116개
4/8 ProductGroup 노드 생성 중...
  ✓ ProductGroup 노드: 17개
5/8 ColourGroup 노드 생성 중...
  ✓ ColourGroup 노드: 48개
6/8 Department 노드 생성 중...
  ✓ Department 노드: 266개
7/8 Section 노드 생성 중...
  ✓ Section 노드: 54개
8/8 GarmentGroup 노드 생성 중...
  ✓ GarmentGroup 노드: 21개

노드 생성 완료!


## 4. 관계 CSV 생성

In [25]:
# 1) Customer -[:PURCHASED]-> Article
print("1/7 PURCHASED 관계 생성 중...")
rels_purchased = transactions[[
    'customer_id',
    'article_id',
    't_dat',
    'price',
    'sales_channel_id'
]].copy()

print(f"  ✓ PURCHASED 관계: {len(rels_purchased):,}개")
rels_purchased.head(3)

1/7 PURCHASED 관계 생성 중...
  ✓ PURCHASED 관계: 500,000개


,customer_id,article_id,t_dat,price,sales_channel_id
31788323,fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20...,898573003,2020-09-22,0.033881,2
31766498,54e8ebd39543b5a4d69c3e7d79977558d2a606e6540ba0...,573085043,2020-09-22,0.033881,2
31766404,544094a3ab237bf18d7bda9c2265218de4320ce795775e...,839332001,2020-09-22,0.067780,1


In [26]:
# 2) Article -[:OF_TYPE]-> ProductType
print("2/7 OF_TYPE 관계 생성 중...")
rels_of_type = articles[['article_id', 'product_type_no']].copy()
print(f"  ✓ OF_TYPE 관계: {len(rels_of_type):,}개")

# 3) Article -[:IN_GROUP]-> ProductGroup
print("3/7 IN_GROUP 관계 생성 중...")
rels_in_group = articles[['article_id', 'product_group_name']].copy()
print(f"  ✓ IN_GROUP 관계: {len(rels_in_group):,}개")

# 4) Article -[:HAS_COLOUR]-> ColourGroup
print("4/7 HAS_COLOUR 관계 생성 중...")
rels_has_colour = articles[['article_id', 'colour_group_code']].copy()
print(f"  ✓ HAS_COLOUR 관계: {len(rels_has_colour):,}개")

# 5) Article -[:IN_DEPARTMENT]-> Department
print("5/7 IN_DEPARTMENT 관계 생성 중...")
rels_in_department = articles[['article_id', 'department_no']].copy()
print(f"  ✓ IN_DEPARTMENT 관계: {len(rels_in_department):,}개")

# 6) Article -[:IN_SECTION]-> Section
print("6/7 IN_SECTION 관계 생성 중...")
rels_in_section = articles[['article_id', 'section_no']].copy()
print(f"  ✓ IN_SECTION 관계: {len(rels_in_section):,}개")

# 7) Article -[:IN_GARMENT_GROUP]-> GarmentGroup
print("7/7 IN_GARMENT_GROUP 관계 생성 중...")
rels_in_garment_group = articles[['article_id', 'garment_group_no']].copy()
print(f"  ✓ IN_GARMENT_GROUP 관계: {len(rels_in_garment_group):,}개")

print("\n관계 생성 완료!")

2/7 OF_TYPE 관계 생성 중...
  ✓ OF_TYPE 관계: 22,694개
3/7 IN_GROUP 관계 생성 중...
  ✓ IN_GROUP 관계: 22,694개
4/7 HAS_COLOUR 관계 생성 중...
  ✓ HAS_COLOUR 관계: 22,694개
5/7 IN_DEPARTMENT 관계 생성 중...
  ✓ IN_DEPARTMENT 관계: 22,694개
6/7 IN_SECTION 관계 생성 중...
  ✓ IN_SECTION 관계: 22,694개
7/7 IN_GARMENT_GROUP 관계 생성 중...
  ✓ IN_GARMENT_GROUP 관계: 22,694개

관계 생성 완료!


## 5. CSV 파일로 저장

In [27]:
output_dir = './output'
os.makedirs(output_dir, exist_ok=True)

print("CSV 파일 저장 중...")
print("=" * 60)

# 노드 CSV 저장
print("\n[노드 CSV 저장]")
nodes_article.to_csv(f'{output_dir}/nodes_article.csv', index=False)
print(f"  ✓ nodes_article.csv")

nodes_customer.to_csv(f'{output_dir}/nodes_customer.csv', index=False)
print(f"  ✓ nodes_customer.csv")

nodes_product_type.to_csv(f'{output_dir}/nodes_product_type.csv', index=False)
print(f"  ✓ nodes_product_type.csv")

nodes_product_group.to_csv(f'{output_dir}/nodes_product_group.csv', index=False)
print(f"  ✓ nodes_product_group.csv")

nodes_colour_group.to_csv(f'{output_dir}/nodes_colour_group.csv', index=False)
print(f"  ✓ nodes_colour_group.csv")

nodes_department.to_csv(f'{output_dir}/nodes_department.csv', index=False)
print(f"  ✓ nodes_department.csv")

nodes_section.to_csv(f'{output_dir}/nodes_section.csv', index=False)
print(f"  ✓ nodes_section.csv")

nodes_garment_group.to_csv(f'{output_dir}/nodes_garment_group.csv', index=False)
print(f"  ✓ nodes_garment_group.csv")

# 관계 CSV 저장
print("\n[관계 CSV 저장]")
rels_purchased.to_csv(f'{output_dir}/rels_purchased.csv', index=False)
print(f"  ✓ rels_purchased.csv (대용량: {len(rels_purchased):,}개)")

rels_of_type.to_csv(f'{output_dir}/rels_of_type.csv', index=False)
print(f"  ✓ rels_of_type.csv")

rels_in_group.to_csv(f'{output_dir}/rels_in_group.csv', index=False)
print(f"  ✓ rels_in_group.csv")

rels_has_colour.to_csv(f'{output_dir}/rels_has_colour.csv', index=False)
print(f"  ✓ rels_has_colour.csv")

rels_in_department.to_csv(f'{output_dir}/rels_in_department.csv', index=False)
print(f"  ✓ rels_in_department.csv")

rels_in_section.to_csv(f'{output_dir}/rels_in_section.csv', index=False)
print(f"  ✓ rels_in_section.csv")

rels_in_garment_group.to_csv(f'{output_dir}/rels_in_garment_group.csv', index=False)
print(f"  ✓ rels_in_garment_group.csv")

print("\n" + "=" * 60)
print("CSV 파일 저장 완료!")
print(f"저장 위치: {output_dir}/")

CSV 파일 저장 중...

[노드 CSV 저장]
  ✓ nodes_article.csv
  ✓ nodes_customer.csv
  ✓ nodes_product_type.csv
  ✓ nodes_product_group.csv
  ✓ nodes_colour_group.csv
  ✓ nodes_department.csv
  ✓ nodes_section.csv
  ✓ nodes_garment_group.csv

[관계 CSV 저장]
  ✓ rels_purchased.csv (대용량: 500,000개)
  ✓ rels_of_type.csv
  ✓ rels_in_group.csv
  ✓ rels_has_colour.csv
  ✓ rels_in_department.csv
  ✓ rels_in_section.csv
  ✓ rels_in_garment_group.csv

CSV 파일 저장 완료!
저장 위치: ./output/


## 5-1. neo4j-admin import용 CSV 생성

- 노드 CSV: 첫 번째 컬럼에 `:ID` 접미사
- 관계 CSV: `:START_ID`, `:END_ID` 컬럼

In [13]:
admin_dir = './output_admin'
os.makedirs(admin_dir, exist_ok=True)

print("neo4j-admin import용 CSV 생성 중...")
print("=" * 60)

# 노드 CSV (ID 컬럼에 :ID 접미사)
print("\n[노드 CSV 생성]")

# Article 노드
nodes_article_admin = nodes_article.copy()
nodes_article_admin.rename(columns={'article_id': 'article_id:ID(Article)'}, inplace=True)
nodes_article_admin.to_csv(f'{admin_dir}/nodes_article.csv', index=False)
print(f"  ✓ nodes_article.csv")

# Customer 노드
nodes_customer_admin = nodes_customer.copy()
nodes_customer_admin.rename(columns={'customer_id': 'customer_id:ID(Customer)'}, inplace=True)
nodes_customer_admin.to_csv(f'{admin_dir}/nodes_customer.csv', index=False)
print(f"  ✓ nodes_customer.csv")

# ProductType 노드
nodes_product_type_admin = nodes_product_type.copy()
nodes_product_type_admin.rename(columns={'product_type_no': 'product_type_no:ID(ProductType)'}, inplace=True)
nodes_product_type_admin.to_csv(f'{admin_dir}/nodes_product_type.csv', index=False)
print(f"  ✓ nodes_product_type.csv")

# ProductGroup 노드
nodes_product_group_admin = nodes_product_group.copy()
nodes_product_group_admin.rename(columns={'name': 'name:ID(ProductGroup)'}, inplace=True)
nodes_product_group_admin.to_csv(f'{admin_dir}/nodes_product_group.csv', index=False)
print(f"  ✓ nodes_product_group.csv")

# ColourGroup 노드
nodes_colour_group_admin = nodes_colour_group.copy()
nodes_colour_group_admin.rename(columns={'colour_group_code': 'colour_group_code:ID(ColourGroup)'}, inplace=True)
nodes_colour_group_admin.to_csv(f'{admin_dir}/nodes_colour_group.csv', index=False)
print(f"  ✓ nodes_colour_group.csv")

# Department 노드
nodes_department_admin = nodes_department.copy()
nodes_department_admin.rename(columns={'department_no': 'department_no:ID(Department)'}, inplace=True)
nodes_department_admin.to_csv(f'{admin_dir}/nodes_department.csv', index=False)
print(f"  ✓ nodes_department.csv")

# Section 노드
nodes_section_admin = nodes_section.copy()
nodes_section_admin.rename(columns={'section_no': 'section_no:ID(Section)'}, inplace=True)
nodes_section_admin.to_csv(f'{admin_dir}/nodes_section.csv', index=False)
print(f"  ✓ nodes_section.csv")

# GarmentGroup 노드
nodes_garment_group_admin = nodes_garment_group.copy()
nodes_garment_group_admin.rename(columns={'garment_group_no': 'garment_group_no:ID(GarmentGroup)'}, inplace=True)
nodes_garment_group_admin.to_csv(f'{admin_dir}/nodes_garment_group.csv', index=False)
print(f"  ✓ nodes_garment_group.csv")

# 관계 CSV (:START_ID, :END_ID 형식 + ID Space 지정)
print("\n[관계 CSV 생성]")

# PURCHASED 관계
rels_purchased_admin = rels_purchased.copy()
rels_purchased_admin.rename(columns={
    'customer_id': ':START_ID(Customer)',
    'article_id': ':END_ID(Article)'
}, inplace=True)
rels_purchased_admin.to_csv(f'{admin_dir}/rels_purchased.csv', index=False)
print(f"  ✓ rels_purchased.csv")

# OF_TYPE 관계
rels_of_type_admin = rels_of_type.copy()
rels_of_type_admin.rename(columns={
    'article_id': ':START_ID(Article)',
    'product_type_no': ':END_ID(ProductType)'
}, inplace=True)
rels_of_type_admin.to_csv(f'{admin_dir}/rels_of_type.csv', index=False)
print(f"  ✓ rels_of_type.csv")

# IN_GROUP 관계
rels_in_group_admin = rels_in_group.copy()
rels_in_group_admin.rename(columns={
    'article_id': ':START_ID(Article)',
    'product_group_name': ':END_ID(ProductGroup)'
}, inplace=True)
rels_in_group_admin.to_csv(f'{admin_dir}/rels_in_group.csv', index=False)
print(f"  ✓ rels_in_group.csv")

# HAS_COLOUR 관계
rels_has_colour_admin = rels_has_colour.copy()
rels_has_colour_admin.rename(columns={
    'article_id': ':START_ID(Article)',
    'colour_group_code': ':END_ID(ColourGroup)'
}, inplace=True)
rels_has_colour_admin.to_csv(f'{admin_dir}/rels_has_colour.csv', index=False)
print(f"  ✓ rels_has_colour.csv")

# IN_DEPARTMENT 관계
rels_in_department_admin = rels_in_department.copy()
rels_in_department_admin.rename(columns={
    'article_id': ':START_ID(Article)',
    'department_no': ':END_ID(Department)'
}, inplace=True)
rels_in_department_admin.to_csv(f'{admin_dir}/rels_in_department.csv', index=False)
print(f"  ✓ rels_in_department.csv")

# IN_SECTION 관계
rels_in_section_admin = rels_in_section.copy()
rels_in_section_admin.rename(columns={
    'article_id': ':START_ID(Article)',
    'section_no': ':END_ID(Section)'
}, inplace=True)
rels_in_section_admin.to_csv(f'{admin_dir}/rels_in_section.csv', index=False)
print(f"  ✓ rels_in_section.csv")

# IN_GARMENT_GROUP 관계
rels_in_garment_group_admin = rels_in_garment_group.copy()
rels_in_garment_group_admin.rename(columns={
    'article_id': ':START_ID(Article)',
    'garment_group_no': ':END_ID(GarmentGroup)'
}, inplace=True)
rels_in_garment_group_admin.to_csv(f'{admin_dir}/rels_in_garment_group.csv', index=False)
print(f"  ✓ rels_in_garment_group.csv")

print("\n" + "=" * 60)
print("neo4j-admin import용 CSV 저장 완료!")

neo4j-admin import용 CSV 생성 중...

[노드 CSV 생성]
  ✓ nodes_article.csv
  ✓ nodes_customer.csv
  ✓ nodes_product_type.csv
  ✓ nodes_product_group.csv
  ✓ nodes_colour_group.csv
  ✓ nodes_department.csv
  ✓ nodes_section.csv
  ✓ nodes_garment_group.csv

[관계 CSV 생성]
  ✓ rels_purchased.csv
  ✓ rels_of_type.csv
  ✓ rels_in_group.csv
  ✓ rels_has_colour.csv
  ✓ rels_in_department.csv
  ✓ rels_in_section.csv
  ✓ rels_in_garment_group.csv

neo4j-admin import용 CSV 저장 완료!


## 6. 생성된 그래프 구조 요약

In [14]:
print("H&M 지식그래프 구조")
print("=" * 60)

print(f"\n[노드 유형]")
total_nodes = (
    len(nodes_article) +
    len(nodes_customer) +
    len(nodes_product_type) +
    len(nodes_product_group) +
    len(nodes_colour_group) +
    len(nodes_department) +
    len(nodes_section) +
    len(nodes_garment_group)
)
print(f"  • Article: {len(nodes_article):,}개")
print(f"  • Customer: {len(nodes_customer):,}개")
print(f"  • ProductType: {len(nodes_product_type):,}개")
print(f"  • ProductGroup: {len(nodes_product_group):,}개")
print(f"  • ColourGroup: {len(nodes_colour_group):,}개")
print(f"  • Department: {len(nodes_department):,}개")
print(f"  • Section: {len(nodes_section):,}개")
print(f"  • GarmentGroup: {len(nodes_garment_group):,}개")
print(f"  → 총 {total_nodes:,}개 노드")

print(f"\n[관계 유형]")
total_rels = (
    len(rels_purchased) +
    len(rels_of_type) +
    len(rels_in_group) +
    len(rels_has_colour) +
    len(rels_in_department) +
    len(rels_in_section) +
    len(rels_in_garment_group)
)
print(f"  • PURCHASED (Customer → Article): {len(rels_purchased):,}개")
print(f"  • OF_TYPE (Article → ProductType): {len(rels_of_type):,}개")
print(f"  • IN_GROUP (Article → ProductGroup): {len(rels_in_group):,}개")
print(f"  • HAS_COLOUR (Article → ColourGroup): {len(rels_has_colour):,}개")
print(f"  • IN_DEPARTMENT (Article → Department): {len(rels_in_department):,}개")
print(f"  • IN_SECTION (Article → Section): {len(rels_in_section):,}개")
print(f"  • IN_GARMENT_GROUP (Article → GarmentGroup): {len(rels_in_garment_group):,}개")
print(f"  → 총 {total_rels:,}개 관계")

H&M 지식그래프 구조

[노드 유형]
  • Article: 105,542개
  • Customer: 1,371,980개
  • ProductType: 132개
  • ProductGroup: 19개
  • ColourGroup: 50개
  • Department: 299개
  • Section: 57개
  • GarmentGroup: 21개
  → 총 1,478,100개 노드

[관계 유형]
  • PURCHASED (Customer → Article): 31,788,324개
  • OF_TYPE (Article → ProductType): 105,542개
  • IN_GROUP (Article → ProductGroup): 105,542개
  • HAS_COLOUR (Article → ColourGroup): 105,542개
  • IN_DEPARTMENT (Article → Department): 105,542개
  • IN_SECTION (Article → Section): 105,542개
  • IN_GARMENT_GROUP (Article → GarmentGroup): 105,542개
  → 총 32,421,576개 관계
